In [1]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
from torch.nn.parameter import Parameter
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
from transformers import AdamW, BertForSequenceClassification, BertModel, BertTokenizer, get_linear_schedule_with_warmup
from transformers.modeling_outputs import SequenceClassifierOutput, BaseModelOutputWithPoolingAndCrossAttentions
from transformers.models.bert.modeling_bert import BertEncoder
from typing import List, Tuple, Dict, Union

/Users/aladindjuhera/Desktop/resilient_sfl/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Classes

class CustomBertEncoder(BertEncoder):
    """
    Custom BERT Encoder class for the attention layer. 
    """
    def __init__(self, config) -> None:
        super().__init__(config)
    
    def forward(self, 
                inputs_embeds, 
                attention_mask=None, 
                head_mask=None) -> BaseModelOutputWithPoolingAndCrossAttentions:

        hidden_states = inputs_embeds
        
        # Ensure attention mask dtype is the same as hidden_states.dtype and
        # modify attention mask values for masked positions
        if attention_mask is not None:
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
            attention_mask = attention_mask.to(dtype=hidden_states.dtype)

        for layer in self.layer:
            outputs = layer(hidden_states, attention_mask, head_mask)
            hidden_states = outputs[0]
    
        return hidden_states


class CustomBertModel(BertForSequenceClassification):
    """
    Custom BERT Model class for testing SFL.
    """
    def __init__(self, config) -> None:
        super().__init__(config)
        
        # Create separate modules for embedding, attention, and head layers
        self.embedding = BertModel(config)
        self.attention = CustomBertEncoder(config)
        self.head = self.classifier

    
    def forward(self, 
                input_ids, 
                attention_mask=None, 
                token_type_ids=None, 
                position_ids=None, 
                head_mask=None, 
                labels=None) -> Tuple[Union[SequenceClassifierOutput, Tuple[torch.tensor]], torch.tensor]:

        # Embedding layer pass
        embeddings = self.embedding(input_ids=input_ids, 
                                    attention_mask=attention_mask, 
                                    token_type_ids=token_type_ids, 
                                    position_ids=position_ids)[0]
        
        # Attention layer pass
        attentions = self.attention(inputs_embeds=embeddings, 
                                    attention_mask=attention_mask,
                                    head_mask=head_mask)
   
        # Head layer pass
        logits = self.head(attentions[:, 0, :])
        
        # Compute loss if labels are provided
        if labels is not None:
            loss = torch.nn.CrossEntropyLoss()(logits, labels)
            return SequenceClassifierOutput(loss=loss, logits=logits)
        else:
            return logits


class Manager():
    """
    Manager class for testing SFL. Handles model loading, data preprocessing, etc.
    """
    def __init__(self, 
                 name: str = "bert_manager",
                 model_type: str = "bert-base-uncased", 
                 batch_size: int = 256) -> None:
        self.name = name
        self.model_type = model_type
        self.batch_size = batch_size


    def load_model(self, num_labels: int) -> CustomBertModel:
        """
        Loads the specified BERT model.
        """
        self.model = CustomBertModel.from_pretrained(self.model_type, num_labels=num_labels)
        return self.model


    def tokenize(self, dataset) -> torch.utils.data.TensorDataset:
        """
        Tokenizes the dataset.
        """
        self.tokenizer = BertTokenizer.from_pretrained(self.model_type)
        encodings = self.tokenizer(dataset["sentence"], truncation=True, padding=True)
        labels = torch.tensor(dataset["label"], dtype=torch.long)
        input_ids = torch.tensor(encodings['input_ids'])
        attention_mask = torch.tensor(encodings['attention_mask'])
        dataset = torch.utils.data.TensorDataset(input_ids, attention_mask, labels)
        return dataset  


    def preprocess_dataset(self, 
                           glue_dataset: str = "sst2", 
                           truncate: int = None
                           ) -> Tuple[DataLoader, DataLoader]:
        """
        Preprocesses the dataset for SFL: tokenization, truncation, DataLoader
        """

        # Load dataset
        dataset = load_dataset("glue", glue_dataset)
        train_dataset, valid_dataset, test_dataset = dataset["train"], dataset["validation"], dataset["test"]

        # Truncate dataset
        if truncate is not None:
            train_dataset = train_dataset.select(range(truncate))
            valid_dataset = valid_dataset.select(range(truncate))
            test_dataset = test_dataset.select(range(truncate))
        else:
            pass

        # Tokenize dataset
        train_dataset = self.tokenize(train_dataset)
        valid_dataset = self.tokenize(valid_dataset)
        test_dataset = self.tokenize(test_dataset)

        # Create dataloaders
        train_dataloader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        valid_dataloader = DataLoader(valid_dataset, batch_size=self.batch_size, shuffle=True)
        test_dataloader = DataLoader(test_dataset, batch_size=self.batch_size, shuffle=True)

        return train_dataloader, valid_dataloader, test_dataloader


    def split_data(self, 
                   data: DataLoader, 
                   clients: int = 2) -> List[DataLoader]:
        """
        Splits the training and validation datasets into equally sized client portions.
        """
        dataset = data.dataset
        split_size = len(dataset) // clients
        remainder = len(dataset) % clients

        split_lengths = [split_size + 1 if i < remainder else split_size for i in range(clients)]
        splits = random_split(dataset, split_lengths)

        split_dataloaders = [DataLoader(split, batch_size=data.batch_size) for split in splits]

        return split_dataloaders


    def save_plots(self,
                   loss: List[float], 
                   accuracy: List[float],
                   title: str,
                   path= str) -> None:
        """
        Saves the loss and accuracy plots.
        """
        plt.figure(figsize=(10, 5))

        epochs = range(1, len(loss) + 1)

        # Plot the loss values
        plt.subplot(1, 2, 1)
        plt.plot(epochs, loss, 'r', label='Loss')
        plt.title('Training Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()

        # Plot the accuracy values
        plt.subplot(1, 2, 2)
        plt.plot(epochs, accuracy, 'b', label='Training Accuracy')
        plt.title('Training vs. Validation Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.legend()

        # Title
        plt.suptitle(title)

        # Save the plot
        plt.savefig(path)


    def save_metric(self, metric: List[float], path: str) -> None:
        """
        Saves the metric (e.g. loss or accuracy) to the specified path.
        """
        np.array(metric)
        np.save(path, metric)


    def load_metric(self, path: str) -> List[float]:
        """
        Loads the metric (e.g. loss or accuracy) from the specified path.
        """
        metric = np.load(path)
        return metric

    
class Client:
    """
    Client class for SFL.
    """
    def __init__(self, 
                 name: str, 
                 model: BertForSequenceClassification,
                 manager: Manager, 
                 train_data: DataLoader,
                 valid_data: DataLoader) -> None:
        self.name = name
        self.model = model
        self.manager = manager
        self.train_data = train_data
        self.valid_data = valid_data

    
    def save_model(self, path: str) -> None:
        """
        Saves the current model to the specified path.
        """
        torch.save(self.model.state_dict(), path)


    def train_model(self, 
                    device: torch.device, 
                    checkpoint_path: str,
                    num_epochs: int = 1, 
                    learning_rate: float = 1e-5,
                    epsilon: float = 1e-6,
                    num_warmup_steps: float = 1256) -> Tuple[List[float], List[float]]:
        """
        Trains the client model on its data.
        """
        self.model.to(device)

        total_steps = len(self.train_data) * num_epochs

        optimizer = AdamW(self.model.parameters(), lr=learning_rate, eps=epsilon)
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_steps)

        # Training Loop
        global_loss = []
        global_accuracy = []

        for epoch in range(num_epochs):
            print(f"Epoch {epoch + 1}/{num_epochs}")
            print("-" * 10)

            self.model.train()
            
            total_loss = 0
            total_correct = 0
            total_samples = 0
            
            # Training
            for step, batch in enumerate(tqdm(self.train_data)):

                batch = tuple(t.to(device) for t in batch)
                inputs = {
                    "input_ids": batch[0],
                    "attention_mask": batch[1],
                    "labels": batch[2]
                }
                
                outputs = self.model(**inputs)
                loss = outputs.loss
                total_loss += loss.detach().float()

                loss.backward()
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            # Validation
            self.model.eval()

            for step, batch in enumerate(tqdm(self.valid_data)):

                batch = tuple(t.to(device) for t in batch)
                inputs = {
                    "input_ids": batch[0],
                    "attention_mask": batch[1],
                    "labels": batch[2]
                }

                with torch.no_grad():
                    outputs = self.model(**inputs)

                eval_loss = outputs.loss
                logits = outputs.logits

                predictions = torch.argmax(logits, dim=1)
                true_labels = batch[2]

                total_correct += (predictions == true_labels).sum().item()
                total_samples += len(true_labels)
                total_loss += eval_loss.item()

            # Compute average loss and accuracy
            average_loss = total_loss / len(self.train_data)
            accuracy = total_correct / total_samples

            print(f"Average Loss: {average_loss:.4f}")
            print(f"Accuracy: {accuracy:.4f}")

            global_loss.append(average_loss.cpu())
            global_accuracy.append(accuracy)


        # Save model
        # checkpoint_path = f"/home/munichcenter/code/bert/sst2/fedavg/results/bert_results/checkpoints/{self.name}_model.pt"
        checkpoint_path = checkpoint_path + f"/{self.name}_model.pt"
        self.save_model(path=checkpoint_path)

        print("Training complete!")

        return global_loss, global_accuracy


    def evaluate_model(self, 
                       model_path: str, 
                       device: torch.device, 
                       valid_data: DataLoader) -> None:
        """
        Evaluates the client model on its validation data.
        """
        print("START EVALUATION")
        self.valid_data = valid_data
        self.model.load_state_dict(torch.load(model_path))
        self.model.to(device)
        self.model.eval()

        total_loss = 0
        total_correct = 0
        total_samples = 0

        for step, batch in enumerate(tqdm(self.valid_data)):

            batch = tuple(t.to(device) for t in batch)
            inputs = {
                "input_ids": batch[0],
                "attention_mask": batch[1],
                "labels": batch[2]
            }

            with torch.no_grad():
                outputs = self.model(**inputs)

            eval_loss = outputs.loss
            logits = outputs.logits

            predictions = torch.argmax(logits, dim=1)
            true_labels = batch[2]

            total_correct += (predictions == true_labels).sum().item()
            total_samples += len(true_labels)
            total_loss += eval_loss.item()

        # Compute average loss and accuracy
        average_loss = total_loss / len(self.valid_data)
        accuracy = total_correct / total_samples

        print(f"Average Loss: {average_loss:.4f}")
        print(f"Accuracy: {accuracy:.4f}")


    def evaluate_global_model(self, 
                              device: torch.device, 
                              valid_data: DataLoader) -> Tuple[float, float]:
        """
        Evaluates a global model (e.g. fedavg-ed) on its validation data and 
        returns the average loss and accuracy.
        """
        print("START EVALUATION")
        self.valid_data = valid_data
        self.model.to(device)
        self.model.eval()

        total_loss = 0
        total_correct = 0
        total_samples = 0

        for step, batch in enumerate(tqdm(self.valid_data)):

            batch = tuple(t.to(device) for t in batch)
            inputs = {
                "input_ids": batch[0],
                "attention_mask": batch[1],
                "labels": batch[2]
            }

            with torch.no_grad():
                outputs = self.model(**inputs)

            eval_loss = outputs.loss
            logits = outputs.logits

            predictions = torch.argmax(logits, dim=1)
            true_labels = batch[2]

            total_correct += (predictions == true_labels).sum().item()
            total_samples += len(true_labels)
            total_loss += eval_loss.item()

        # Compute average loss and accuracy
        average_loss = total_loss / len(self.valid_data)
        accuracy = total_correct / total_samples

        print(f"Average Loss: {average_loss:.4f}")
        print(f"Accuracy: {accuracy:.4f}")

        return average_loss, accuracy


    def update_model(self, updates: Dict[str, Parameter]) -> None:
        """
        Updates the client model with the aggregated weight updates.
        """
        for gradient_name in updates.keys():
            for name, param in self.model.named_parameters():
                if name == gradient_name:
                    param.data = updates[gradient_name]


class Aggregator:
   """
   Aggregator class for prototyping SFL. Handles the aggregation of client gradients.
   """
   def __init__(self, name: str) -> None:
        self.name = name

      
   def accumulate_attentions(self, client_models: List[BertForSequenceClassification]) -> List[Dict]:
    """
    Accumulates/Collects the attentions of all clients and returns a List of 
    Dicts including the parameter names and values as a key-value pair.
    """
    attentions = []

    for model in client_models:
        attention_gradients = {}
        for name, param in model.named_parameters():
            if name.startswith("bert.encoder"):
                attention_gradients[name] = param

        attentions.append(attention_gradients)

    return attentions
   
   
   def accumulate_heads(self, client_models: List[BertForSequenceClassification]) -> List[Dict]:
    """
    Accumulates/Collects the heads (classifier and pooler) of all clients and 
    returns a List of Dicts including the parameter names and values as a 
    key-value pair.
    """
    heads = []

    for model in client_models:
        head_gradients = {}
        for name, param in model.named_parameters():
            if name.startswith("classifier") or name.startswith("bert.pooler"):
                head_gradients[name] = param

        heads.append(head_gradients)

    return heads


   def accumulate_embeddings(self, client_models: List[BertForSequenceClassification]) -> List[Dict]:
    """
    Accumulates/Collects the embeddings of all clients and returns a List of 
    Dicts including the parameter names and values as a key-value pair.
    """
    embeddings = []

    for model in client_models:
        embedding_gradients = {}
        for name, param in model.named_parameters():
            if name.startswith("bert.embeddings"):
                embedding_gradients[name] = param

        embeddings.append(embedding_gradients)

    return embeddings

   
   def aggregate(self, client_gradients: List[Dict[str, Parameter]]) -> Dict[str, Parameter]:
      """
      Aggregates the parameters of the clients by simple averaging.
      """
      aggregated_gradients = {}
      num_clients = len(client_gradients)

      for client_gradient in client_gradients:
         # Iterate over each parameter gradient in the client's gradients
         for param_name, gradient in client_gradient.items():
               if param_name not in aggregated_gradients:
                  # Initialize the tensor to store the average
                  aggregated_gradients[param_name] = torch.zeros_like(gradient.data)
               
               # Accumulate the gradient values across all clients
               aggregated_gradients[param_name] += gradient.data

      # Compute the average for each parameter gradient
      for param_name in aggregated_gradients:
         aggregated_gradients[param_name] /= num_clients

      return aggregated_gradients

In [ ]:
# Hyperparameters
num_rounds = 2
num_epochs = 2
num_clients = 2

# Instantiate manager
manager = Manager(name="bert-manager", model_type="bert-base-uncased", batch_size=16)

# Load data
train_dataloader, valid_dataloader, test_dataloader = manager.preprocess_dataset(glue_dataset="sst2", truncate=100)
train_split = manager.split_data(train_dataloader, clients=num_clients)
valid_split = manager.split_data(valid_dataloader, clients=num_clients)
test_split = manager.split_data(test_dataloader, clients=num_clients)

# Load model
num_labels = len(set(train_dataloader.dataset.tensors[2].tolist()))
pretrained_model = manager.load_model(num_labels=num_labels)

In [ ]:
# Global training
global_loss = []
global_accuracy = []
dir = os.getcwd()

for r in range(num_rounds):

    print(f"GLOBAL ROUND : {r+1} of {num_rounds}")

    # Instantiate clients
    for i in range(num_clients):
        client = Client(name=f"client_{i}",
                        model=pretrained_model,
                        manager=manager,
                        train_data=train_split[i],
                        valid_data=valid_split[i])
        
        if r != 0:  
            # Load model from path
            print("Loading Model from Directory...")
            checkpoint_path = dir + f"/checkpoints/{client.name}_model.pt"
            checkpoint = torch.load(checkpoint_path)
            client.model.load_state_dict(checkpoint)
        else:
            pass

        # Local training
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        client.train_model(device=device, num_epochs=num_epochs, checkpoint_path=dir+"/checkpoints")
    
    # Load client models from path
    clients = []
    for i in range(num_clients):
        client = Client(name=f"client_{i}",
                        model=pretrained_model,
                        manager=manager,
                        train_data=train_split[i],
                        valid_data=valid_split[i])

        checkpoint_path = dir + f"/checkpoints/{client.name}_model.pt"
        checkpoint = torch.load(checkpoint_path)
        client.model.load_state_dict(checkpoint)

        clients.append(client)

    # Parameter aggregation
    aggregator = Aggregator(name="bert_aggregator")

    attentions = aggregator.accumulate_attentions([client.model for client in clients])
    heads = aggregator.accumulate_heads([client.model for client in clients])
    embeddings = aggregator.accumulate_embeddings([client.model for client in clients])

    aggregated_attentions = aggregator.aggregate(attentions)
    aggregated_heads = aggregator.aggregate(heads)
    aggregated_embeddings = aggregator.aggregate(embeddings)

    # Model update and save
    for client in clients:
        client.update_model(aggregated_attentions)
        client.update_model(aggregated_heads)
        client.update_model(aggregated_embeddings)

        model_path = dir + f"/checkpoints/{client.name}_model.pt"
        client.save_model(path=model_path)

    # Track global model performance
    global_model = clients[-1]
    loss, acc = global_model.evaluate_global_model(device=device, valid_data=valid_dataloader) 
    global_loss.append(loss)
    global_accuracy.append(acc)
    

# Save final model
final_model = clients[-1].model
final_model_path = dir + "/checkpoints/fedavg_bert_sst2_model.pt"
torch.save(final_model.state_dict(), final_model_path)

In [ ]:
# Plot global model performance
manager.save_plots(loss=global_loss, 
                   accuracy=global_accuracy, 
                   title="FedAvg BERT Training",
                   path=dir + "/plots/fedavg_bert_sst2.png")

In [ ]:
# Save global model metrics
metric_path = dir + "/metrics/"
manager.save_metric(metric=global_loss, path=metric_path+"global_loss")
manager.save_metric(metric=global_accuracy, path=metric_path+"global_accuracy")

In [ ]:
# Retrieve model metrics
loss_metric_path = metric_path + "global_loss.npy"
accuracy_metric_path = metric_path + "global_accuracy.npy"

global_loss = np.load(loss_metric_path)
global_accuracy = np.load(accuracy_metric_path)

print(global_loss)
print("--" * 10)
print(global_accuracy)

In [ ]:
# Evaluate client from directory
client = Client(name="eval_client",
                model=pretrained_model,
                manager=manager,
                train_data=train_split[0],
                valid_data=valid_split[0])

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_path = dir + "/checkpoints/fedavg_bert_sst2_model.pt"
client.evaluate_model(model_path=model_path, device=device, valid_data=valid_dataloader)